In [2]:
from scrapy.spiders import CrawlSpider, Rule
from scrapy.item import Item, Field
from scrapy.selector import Selector
from scrapy.loader import ItemLoader
from scrapy.linkextractors import LinkExtractor
from scrapy.crawler import CrawlerProcess
from scrapy.loader.processors import MapCompose as mc
import os

os.environ.setdefault('SCRAPY_SETTINGS_MODULE', 'config.settings')

'config.settings'

In [4]:
class Product(Item):
    title = Field()
    description = Field()
    price = Field()
    amount_reviews = Field()


class Reviews(Item):
    score = Field()
    reviews = Field()


class Amazon(CrawlSpider):
    Name = "Beauty Products"
    custom_settings = {
        "USER_AGENT": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36",
        "CLOSESPIDER_ITEMCOUNT": 20,
        "CLOSESPIDER_COUNTPAGE": 10,
    }

    allowed_domains = ["www.amazon.com"]

    start_urls = ["https://www.amazon.com/s?k=beauty+products"]

    download_delay = 2

    visited_urls = set()

    rules = (
        # product details
        Rule(
            LinkExtractor(
                allow=r"/dp/B0\w+"
            ), follow=True, callback="to_get_products"
        ),
        # Product pagination
        Rule(
            LinkExtractor(
                allow=r"&page=\d+&"
            )
        ),
        ## get reviews
        Rule(
            LinkExtractor(
                allow=r"/customer-reviews/"
            ), follow=True, callback="to_get_review"
        )
    )

    def to_get_products(self, response):
        # verificar si la url ya fue visitada
        if response.url in self.visited_urls:
            self.logger.warning(f"Loop Dectectado: {response.url}")
            return # no procesar si es recursiva

        # registrar la url como visitada
        self.visited_urls.add(response.url)

        item = ItemLoader(Product(), response)

        item.add_xpath('title', '')
        item.add_xpath('description', '')
        item.add_xpath('price', '')
        item.add_xpath('amount_reviews', '')


    def to_get_reviews(self, response):
        # verificar si la url ya fue visitada
        if response.url in self.visited_urls:
            self.logger.warning(f"Loop Detectado: {response.url}")
            return # no procesar si es recursiva

        # registrar la url como visitada
        self.visited_urls.add(response.url)

        item = ItemLoader(Reviews(), response)

        item.add_xpath('score', '')
        item.add_xpath('reviews', '')

        

## possible pattern dp/B0B2RM68G2
    